# C2.1 Production RAG Architecture

This lesson is production-first: decide when to use RAG, build a corpus, evaluate chunking and retrieval, and assemble a grounded RAG pipeline.

Learning goals:
- Choose RAG vs fine-tuning with clear tradeoffs.
- Compare chunking strategies with retrieval metrics.
- Implement query rewriting and measure its impact.
- Build a complete RAG pipeline without external downloads.

## RAG vs fine-tuning decision

**Direct answer for this program:** use RAG for knowledge-grounded applications. Fine-tuning changes behavior, not knowledge.

Why RAG dominates production for knowledge-grounded apps:
- Retrieves fresh, private, and auditable data at request time.
- Keeps model weights stable while knowledge changes frequently.
- Reduces hallucinations by grounding responses in retrieved evidence.

Why fine-tuning is often impractical for product teams without training infra:
- Requires labeled data, GPUs, and training pipelines.
- Updates are slow and expensive when knowledge changes.
- Hard to trace or cite sources for answers.

Tradeoffs (cost, maintenance, recency):

| Dimension | RAG | Fine-tuning |
| --- | --- | --- |
| Cost | Low upfront, higher per-query | High upfront, lower per-query |
| Maintenance | Update index quickly | Retrain to update knowledge |
| Recency | Immediate | Delayed until retrain |

## Chunking strategies

No single method is universally best. Choose based on data type, retrieval goals, and system architecture.

Typical fits:
- Sentence-aware (recursive/sentence-level) for general-purpose RAG when you want coherent context without heavy compute.
- Semantic chunking for topic-heavy, long documents where precision matters and extra compute is acceptable.
- Fixed-size (sliding window) for high-throughput pipelines where predictability matters most.

Detailed breakdown:

1. Sentence-aware (recursive/sentence-level)
- What it does: splits text by logical units like sentences or paragraphs.
- Best fit: standard RAG use cases that need coherent context.
- Pros: preserves complete thoughts and avoids mid-sentence breaks.
- Cons: inconsistent chunk sizes can affect models expecting uniform inputs.

2. Semantic chunking
- What it does: uses embeddings or NLP similarity to split when topics shift.
- Best fit: long research papers, dense manuals, or multi-topic documents.
- Pros: high retrieval precision because chunks stay concept-cohesive.
- Cons: more complex and compute-heavy because it requires embedding to chunk.

3. Fixed size (sliding window)
- What it does: splits at a fixed token or character count with overlap.
- Best fit: logs, structured reports, or large datasets where throughput matters.
- Pros: simple, fast, and predictable latency.
- Cons: breaks sentences and can dilute semantic meaning.

Tradeoff snapshot (no universal winner):

| Method | Coherence | Compute | Latency predictability | Typical fit |
| --- | --- | --- | --- | --- |
| Sentence-aware | High | Medium | Medium | General RAG |
| Semantic | Highest | High | Low | Topic-heavy docs |
| Fixed-size | Low to Medium | Low | High | High-throughput |

Chunk size tradeoff:
- Too small: loses context and harms retrieval quality.
- Too large: mixes topics and adds irrelevant context.
- Balanced sizes preserve meaning and keep retrieval precise.

In [39]:
from pathlib import Path
import csv
import json
import math
import re

import numpy as np

import sys
sys.path.append('Track2/Functions')
from C2_1_Func import (
    build_corpus_documents, persist_corpus, load_eval_queries,
    load_txt, load_md, load_jsonl, load_csv, load_corpus,
    tokenize, SimpleTfidfVectorizer, split_sentences, fixed_size_chunking,
    sentence_aware_chunking, cosine_similarity, semantic_chunking,
    build_chunks, cosine_similarity_scores, retrieve_top_k, evaluate_chunking,
    BM25, bm25_retrieve, retrieve_with_threshold, eval_threshold, eval_top_k,
    eval_retrieval_method, lexical_search, run_retrieval, rewrite_expansion,
    rewrite_relaxation, rewrite_segmentation, rewrite_scoping, evaluate_lexical,
)

DATA_DIR = Path('Track2') / 'demos' / 'data' / 'C2.1_corpus'

In [ ]:
documents = build_corpus_documents()
persist_corpus(documents, DATA_DIR)
corpus_documents = documents
corpus_queries = load_eval_queries(DATA_DIR)
EVAL_DOC_IDS = {row['doc_id'] for row in corpus_queries}

fmt_counts = {}
for doc in documents:
    fmt_counts[doc['format']] = fmt_counts.get(doc['format'], 0) + 1

topic_counts = {}
for doc in documents:
    topic_counts[doc['topic']] = topic_counts.get(doc['topic'], 0) + 1

print('Corpus ready.')
print(f'Total documents : {len(documents)}')
print(f'By format       : {fmt_counts}')
print(f'By domain       : {topic_counts}')
print(f'Eval query set  : {len(corpus_queries)} queries across {len(EVAL_DOC_IDS)} documents')
print(f'Sample (finance): "{documents[3]["title"]}" - Q: {documents[3]["query"]}')

60 domain-rich documents across Finance (15), Education (15), Healthcare (10), Technology (10), AI/ML (5), and Legal (5). Each document is 3–4 realistic sentences drawn from practitioner knowledge in that field. The 16-query evaluation set spans all six domains so learners from any background can observe retrieval working on content they recognize.

In [41]:
print("Multi-format loaders (load_txt, load_md, load_jsonl, load_csv, load_corpus) loaded from C2_1_Func.")

In [42]:
ingested_docs = load_corpus(DATA_DIR)

counts = {}
for doc in ingested_docs:
    counts[doc['source']] = counts.get(doc['source'], 0) + 1

print(f'Ingested documents: {len(ingested_docs)}')
print(f'By format: {counts}')
print(f"Example title: {ingested_docs[0]['title']}")

Ingested documents: 60
By format: {'txt': 15, 'md': 15, 'jsonl': 15, 'csv': 15}
Example title: Backpropagation and gradients


Ingestion loaded all 60 documents across four formats, and the sample title shows the parser is reading each file type correctly.

## Embedding model selection and retrieval measurement harness

**Embedding model selection**

| Model | Strengths | When to choose |
| --- | --- | --- |
| OpenAI `text-embedding-3-small/large` | Strong English quality, easy API integration | Default for English-first products where cost per query is acceptable |
| Cohere `embed-multilingual-v3` | Strong multilingual and domain-adaptation support | Multi-language corpora or messy real-world text |
| Open-source (BGE, E5, sentence-transformers) | No API dependency, offline use, data sovereignty | Privacy-sensitive data, high throughput, cost control |

**Retrieval patterns**

- **Top-k:** always returns exactly k results regardless of relevance score. Safe when you need a fixed-size context window for the LLM.
- **Similarity threshold:** discards results below a quality cutoff. Reduces noise but may return zero results on ambiguous queries.

**Measurement harness** — the cells below compare TF-IDF and BM25 across three metrics:
- **Top-1 accuracy:** correct answer appears in the first result.
- **Top-3 accuracy:** correct answer appears anywhere in the top 3 results.
- **MRR (Mean Reciprocal Rank):** average of 1/rank for each query; higher is better, max 1.0.

**Hallucination reduction through grounding** — when the LLM answer is assembled only from retrieved chunks, errors are anchored to the corpus. Without grounding, the model fills gaps with plausible-sounding but fabricated text. The hallucination demo below shows this difference directly.

In [46]:
threshold_chunker = lambda text: sentence_aware_chunking(text, max_words=40, overlap_sentences=1)

threshold_chunks = build_chunks(ingested_docs, threshold_chunker)
threshold_vectorizer = SimpleTfidfVectorizer()
threshold_vectors = threshold_vectorizer.fit_transform([c['text'] for c in threshold_chunks])

topk_hit, topk_avg = eval_top_k(3, corpus_queries, threshold_chunks, threshold_vectorizer, threshold_vectors)
threshold_hit, threshold_avg = eval_threshold(0.20, corpus_queries, threshold_chunks, threshold_vectorizer, threshold_vectors)

print('Retrieval pattern comparison')
print('Pattern | Hit rate | Avg returned')
print(f'Top-k (k=3) | {topk_hit:.2f} | {topk_avg:.2f}')
print(f'Threshold (0.20) | {threshold_hit:.2f} | {threshold_avg:.2f}')

Retrieval pattern comparison
Pattern | Hit rate | Avg returned
Top-k (k=3) | 1.00 | 3.00
Threshold (0.20) | 1.00 | 2.00


In [ ]:
# Build a sentence-aware index for TF-IDF baseline
harness_chunker = lambda text: sentence_aware_chunking(text, max_words=50, overlap_sentences=1)
harness_chunks  = build_chunks(ingested_docs, harness_chunker)
tfidf_vec       = SimpleTfidfVectorizer()
tfidf_vecs      = tfidf_vec.fit_transform([c['text'] for c in harness_chunks])

tfidf_retrieve_fn = lambda query, top_k=3: retrieve_top_k(query, harness_chunks, tfidf_vec, tfidf_vecs, top_k=top_k)

# Build a BM25 index on the same chunks
bm25 = BM25(k1=1.5, b=0.75).fit([c['text'] for c in harness_chunks])
bm25_retrieve_fn = lambda query, top_k=3: bm25_retrieve(query, bm25, harness_chunks, top_k=top_k)

tfidf_result = eval_retrieval_method('TF-IDF',  tfidf_retrieve_fn, corpus_queries)
bm25_result  = eval_retrieval_method('BM25',    bm25_retrieve_fn,  corpus_queries)

print('Retrieval model comparison - 16-query multi-domain eval set')
print(f'{"Method":<10} | Top1 | Top3 |  MRR | Delta vs TF-IDF')
for r in [tfidf_result, bm25_result]:
    if r is tfidf_result:
        delta = '---'
    else:
        delta = (f"{r['top1']-tfidf_result['top1']:+.2f}/"
                 f"{r['top3']-tfidf_result['top3']:+.2f}/"
                 f"{r['mrr']-tfidf_result['mrr']:+.2f}")
    print(f"{r['method']:<10} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f} | {delta}")

print()
print('Interpretation:')
print('  TF-IDF weights terms by corpus frequency but treats all term occurrences equally.')
print('  BM25 saturates term frequency and normalises by doc length, which tends to')
print('  improve precision for short factual queries like the ones in this eval set.')
print('  In practice, production RAG systems use embedding models (dense retrieval)')
print('  which further outperform sparse methods, especially for paraphrased queries.')

In [47]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.40]

print('Threshold sweep')
print('Threshold | Hit rate | Avg returned')
for t in thresholds:
    hit_rate, avg_returned = eval_threshold(t, corpus_queries, threshold_chunks, threshold_vectorizer, threshold_vectors)
    print(f'{t:>8.2f} | {hit_rate:.2f} | {avg_returned:.2f}')

Threshold sweep
Threshold | Hit rate | Avg returned
    0.05 | 1.00 | 4.42
    0.10 | 1.00 | 3.50
    0.20 | 1.00 | 2.00
    0.30 | 1.00 | 1.00
    0.40 | 1.00 | 1.00


As the threshold rises, fewer chunks are returned while the hit rate stays at 1.00 here. This illustrates the precision-recall tradeoff: higher thresholds reduce noise but can risk missing answers on tougher corpora.
The threshold filter keeps the hit rate at 1.00 while returning fewer chunks on average (2 vs 3), showing how similarity thresholds can cut noise without sacrificing coverage at a well-chosen cutoff.

## Query rewriting

There is no single best technique. Each method supports a different precision vs recall goal, and real systems often combine them. In the lab below, each method is compared against its own baseline (no rewrite) to show improvement rather than to rank methods.

1. Query segmentation
- What it does: breaks a query into semantic phrases.
- Best used for: improving precision by treating key phrases as a unit.

2. Query scoping
- What it does: restricts matching to specific fields (title, tags, or metadata).
- Best used for: aggressive precision to avoid irrelevant matches.

3. Query relaxation
- What it does: removes optional terms when results are too narrow.
- Best used for: increasing recall so users still get results.

4. Query expansion
- What it does: adds synonyms or related terms to reduce vocabulary mismatch.
- Best used for: improving recall when the corpus uses different wording.

Recommended pipeline:
- Segment first so the system understands core phrases.
- Scope second to enforce where those phrases must match.
- Expand or relax last if results are too narrow.

In [ ]:
# Ordered to surface baseline errors for segmentation/scoping.
rewrite_docs = {
    1: {'title': 'Intellectual Property Attorney', 'body': 'An attorney specializing in intellectual property law.'},
    2: {'title': 'IP Networking Basics', 'body': 'Internet protocol and computer networking concepts.'},
    3: {'title': 'Cute Fluffy Kittens', 'body': 'Photos of fluffy kittens and cute cats.'},
    4: {'title': 'Fluffy Kittens Playing', 'body': 'Small fluffy kittens playing together.'},
    6: {'title': 'White Shirt Dress', 'body': 'Elegant white shirt dress for women.'},
    5: {'title': 'White Dress Shirt', 'body': 'Formal white dress shirt for office wear.'},
    9: {'title': 'Artificial Intelligence', 'body': 'Machine learning is widely used in AI.'},
    7: {'title': 'Machine Learning', 'body': 'Introduction to machine learning algorithms.'},
    8: {'title': 'Learning Techniques', 'body': 'Different educational learning methods.'},
    10: {'title': 'Dress Collection', 'body': 'Women dress collection and fashion.'},
    11: {'title': 'Cute Puppies', 'body': 'Cute dogs and puppies.'}
}

expansion_queries = [
    {'query': 'ip lawer', 'doc_id': 1},
    {'query': 'ml algorithms', 'doc_id': 7}
]
relaxation_queries = [
    {'query': 'cute fluffy kittens', 'doc_id': 3},
    {'query': 'tiny fluffy kittens playing', 'doc_id': 4}
]
segmentation_queries = [
    {'query': 'white dress shirt', 'doc_id': 5}
]
scoping_queries = [
    {'query': 'machine learning', 'doc_id': 7},
    {'query': 'learning techniques', 'doc_id': 8}
]

SYNONYMS = {
    'ip': ['intellectual property'],
    'lawer': ['lawyer', 'attorney'],
    'ml': ['machine learning']
}
RELAX_TERMS = {'tiny'}
PHRASES = ['dress shirt', 'intellectual property', 'machine learning']

In [ ]:
groups = [
    ('Expansion', expansion_queries, lambda q: {'terms': tokenize(q), 'strict_and': True}, lambda q: rewrite_expansion(q, SYNONYMS)),
    ('Relaxation', relaxation_queries, lambda q: {'terms': tokenize(q), 'strict_and': True}, lambda q: rewrite_relaxation(q, RELAX_TERMS)),
    ('Segmentation', segmentation_queries, lambda q: {'terms': tokenize(q), 'strict_and': False}, lambda q: rewrite_segmentation(q, PHRASES)),
    ('Scoping', scoping_queries, lambda q: {'terms': tokenize(q), 'strict_and': False}, rewrite_scoping)
]

print('Query rewriting evaluation')
print('Method | Top1 before | Top1 after | Top3 before | Top3 after')
for label, queries, baseline_fn, rewrite_fn in groups:
    before = evaluate_lexical(queries, rewrite_docs, baseline_fn)
    after = evaluate_lexical(queries, rewrite_docs, rewrite_fn)
    print(f"{label:<11} | {before['top1']:.2f} | {after['top1']:.2f} | {before['top3']:.2f} | {after['top3']:.2f}")

Query rewriting evaluation
Method | Top1 before | Top1 after | Top3 before | Top3 after
Expansion   | 0.00 | 1.00 | 0.00 | 1.00
Relaxation  | 0.50 | 1.00 | 0.50 | 1.00
Segmentation | 0.00 | 1.00 | 1.00 | 1.00
Scoping     | 0.50 | 1.00 | 1.00 | 1.00


Each method improves over its own baseline, not over the other methods. Expansion and relaxation primarily boost recall, while segmentation and scoping tighten precision when queries are ambiguous.